In [1]:
"""Model Hyperparameters:
Learning Rate: 1*10-4
Batch Size: 4
Number of Epochs: 8
Test Size: 0.4
Max Input Length: 512
Max Target Length: 64

Post Tuning Hyperparameters:
Max Beam: 32
Number of Beams: 8
"""

'Model Hyperparameters:\nLearning Rate: 1*10-4\nBatch Size: 4\nNumber of Epochs: 8\nTest Size: 0.4\nMax Input Length: 512\nMax Target Length: 64\n\nPost Tuning Hyperparameters:\nMax Beam: 32\nNumber of Beams: 8\n'

In [2]:
import os

# Tell transformers not to use TensorFlow (we only need PyTorch here)
os.environ["TRANSFORMERS_NO_TF"] = "1"

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

import transformers, sys, importlib

print("Python executable:", sys.executable)
print("Transformers version:", transformers.__version__)
print("TensorFlow present?", importlib.util.find_spec("tensorflow") is not None)
print("Keras present?", importlib.util.find_spec("keras") is not None)
print("tf_keras present?", importlib.util.find_spec("tf_keras") is not None)

/Users/chadadelman/anaconda3/envs/chad_env/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Python executable: /Users/chadadelman/anaconda3/envs/chad_env/bin/python
Transformers version: 4.57.3
TensorFlow present? True
Keras present? True
tf_keras present? True


In [3]:
#We will need the training data text summary pairs
%store -r train_paired_summaries


In [4]:
#create object to be passed in as batch
#Needs to be dictinionary of lists
#Will be only training data
text_list=[]
summary_list=[]

for v in train_paired_summaries.values():
    text_list.append(v[0])
    summary_list.append(v[1])

training_batch = dict()
training_batch["text"] = text_list
training_batch["summary"] = summary_list

training_batch_dataset = Dataset.from_dict(training_batch)

In [5]:
import torch
from torch.utils.data import DataLoader

def train_model(learning_rate = 5e-7
                ,batch_size = 2
                ,num_epochs = 2
                ,test_size = 0.4
                ,max_input_length = 64
                ,max_target_length = 32
                ,display_progress = True):
    
    # Simple train/validation split
    dataset = training_batch_dataset.train_test_split(test_size=test_size, seed=42)
    train_dataset = dataset["train"]
    eval_dataset = dataset["test"]

    model_name = "t5-small"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    print("Tokenizer type:", type(tokenizer))
    print("Model type:", type(model))

    def preprocess_function(batch):
        # T5 likes a task prefix, e.g. "summarize: "
        inputs = ["summarize: " + t for t in batch["text"]]
        model_inputs = tokenizer(
            inputs,
            max_length=max_input_length,
            truncation=True,
            padding="max_length",
        )

        labels = tokenizer(
            batch["summary"],
            max_length=max_target_length,
            truncation=True,
            padding="max_length",
        )

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    tokenized_train = train_dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=["text", "summary"],
    )

    tokenized_eval = eval_dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=["text", "summary"],
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # DataLoader for training
    train_dataloader = DataLoader(
        tokenized_train,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=data_collator,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    model.train()
    for epoch in range(num_epochs):
        total_loss = 0.0
        for step, batch in enumerate(train_dataloader):
            # Move batch to device
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if display_progress and (step + 1) % 25 == 0:
                print(f"Epoch {epoch+1}, Step {step+1}, Loss: {loss.item():.4f}")

        avg_loss = total_loss / len(train_dataloader)
        
        print(f"Epoch {epoch+1} finished. Average loss: {avg_loss:.4f}")

    model.eval()
    return (model, tokenizer, device)
    


In [ ]:
#New Model 
#Forgot too change variable name but already ran model so this is what it will be called
train_model_config_1 = train_model(learning_rate = 1e-4
                                ,batch_size = 4
                                ,num_epochs = 8
                                ,test_size = 0.4
                                ,max_input_length = 512
                                ,max_target_length = 64
                                ,display_progress = False)


Tokenizer type: <class 'transformers.models.t5.tokenization_t5_fast.T5TokenizerFast'>
Model type: <class 'transformers.models.t5.modeling_t5.T5ForConditionalGeneration'>


Map:   0%|          | 0/454 [00:00<?, ? examples/s]

Map:   0%|          | 0/303 [00:00<?, ? examples/s]

Epoch 1 finished. Average loss: 4.3313
Epoch 2 finished. Average loss: 3.8713
Epoch 3 finished. Average loss: 3.6997
Epoch 4 finished. Average loss: 3.5714
Epoch 5 finished. Average loss: 3.4631
Epoch 6 finished. Average loss: 3.3702
Epoch 7 finished. Average loss: 3.2712
Epoch 8 finished. Average loss: 3.2134


In [7]:
import torch
from torch.utils.data import DataLoader

def run_model(model
              ,tokenizer
              ,device
              ,test_text_list
              ,max_length = 32
              ,num_beams = 2):
    
    generated_text_list=[]

    for test_text in test_text_list:
        #summarize each item in test_text_list
        inputs = tokenizer(
            "summarize: " + test_text,
            return_tensors="pt",
            truncation=True,
            padding=True,
        ).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_length=max_length,
                num_beams=num_beams,
            )

        generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        generated_text_list.append(generated_text)
    
    return generated_text_list


In [9]:
%store -r val_paired_summaries


In [ ]:
val_2_list=list()

#For inital check (2)
val_2_list.append(val_paired_summaries["Hamlet-Act I-Scene I"][0])
val_2_list.append(val_paired_summaries["Sonnet 101"][0])


In [19]:
#Run model on inital check
generated_2_val_list = run_model(model = train_model_config_1[0]
                    ,tokenizer = train_model_config_1[1]
                    ,device = train_model_config_1[2]
                    ,test_text_list = val_2_list
                    ,max_length = 48
                    ,num_beams = 6)

In [20]:
#Looking at data for inital check (looks good)
print(generated_2_val_list)

['Francisco and Barnardo, both sentinels, await their arrival. Barnardo and Horatio await their return to the Dane. They await their return to the Dane, believing he is in danger. The', 'Muse teaches thee how to make him appear, long hence, as he prepares to be praised of ages yet to be. Muse teaches thee how to make him appear, long hence, as']


In [21]:
val_full_list = list()
for v in val_paired_summaries.values():
    val_full_list.append(v[0])

In [22]:
#Looking at data
print(len(val_2_list))
print(len(val_full_list))

2
136


In [23]:
#Run model
generated_full_val_list = run_model(model = train_model_config_1[0]
                    ,tokenizer = train_model_config_1[1]
                    ,device = train_model_config_1[2]
                    ,test_text_list = val_full_list
                    ,max_length = 48
                    ,num_beams = 6)

In [24]:
#Examining results 
print(generated_full_val_list[0])
print(generated_full_val_list[5])
print(generated_full_val_list[23])
print(generated_full_val_list[29])
print(generated_full_val_list[135])

Francisco and Barnardo, both sentinels, await their arrival. Barnardo and Horatio await their return to the Dane. They await their return to the Dane, believing he is in danger. The
Polonius and his lord Reynaldo investigate Reynaldo’s behaviour. Reynaldo prepares for a meeting with Polonius, believing he is incontine
Montano and two Gentlemen discuss the enraged conflict between the Turks and the Turks. They discuss the conflict between the Turks and the Turks. They discuss the conflict between the Turks and the Turk
Desdemona, Emilia, and Clown discuss Cassio’s disappearance. Emilia and Emilia discuss Cassio’s disappearance. Emilia and Emilia discuss Cassio’s
Cupid, a maid of Dian, reveals his love-kindling fire in a cold valley-fountain of that ground. The boy for trial needs to touch his breast, and he


In [ ]:
#store final validation list
%store generated_full_val_list

Stored 'generated_full_val_list' (list)


In [26]:
#score model since it is deemed good and wiss thus be used for testing
%store train_model_config_1

Stored 'train_model_config_1' (tuple)
